# BabyCare AI: Agentic RAG for Pediatric Decision Support

BabyCare AI is a local clinical decision-support workflow for children aged **0 to 60 months**. It applies deterministic safety rules before retrieval, then uses a hand-compiled knowledge graph, TF-IDF evidence retrieval, evidence verification, and optional Claude support.

**Safety statement:** This notebook is for clinician decision support only. It does not diagnose, does not prescribe, does not calculate probabilities, and does not replace clinical judgement. Evidence is retrieved only from indexed local documents. Missing information is never assumed normal.

## Workflow
1. Read the source manifest and index local guideline documents.
2. Build a local TF-IDF retrieval index.
3. Inspect the hand-compiled disease graph and investigation table.
4. Run deterministic safety rules before any retrieval or LLM call.
5. Run the agent workflow on three demonstration cases.
6. Apply follow-up answers and rerun the assessment.
7. Run safety and reference-audit assertions.

All engine functions are imported from `babycare_engine.py`, the same module used by `app.py`.

In [ ]:
import json
from pathlib import Path

import pandas as pd
from IPython.display import Markdown, display

import babycare_engine as be

pd.set_option("display.max_colwidth", 90)
print("Working directory:", Path.cwd())
print("Engine module:", Path(be.__file__).name)
assert Path("babycare_engine.py").exists(), "Run this notebook from the project root."
assert Path("data/sources.csv").exists(), "Create data/sources.csv first."
print("Constants loaded. Disclaimer:", be.DISCLAIMER)

## 1. Source manifest

The manifest records the configured documents and their provenance. Failed downloads remain visible as failures; they are never treated as indexed evidence.

In [ ]:
sources = be.load_sources()
source_table = pd.DataFrame(sources)[["id", "organization", "title", "date", "doc_type"]]
display(source_table)
print(f"{len(sources)} configured sources")

## 2. Download and index documents

The ingestion step extracts text from PDF, HTML, and TXT files, splits it into overlapping chunks, and records an OK/FAIL report. Existing cached downloads are reused. The 2023 Rwanda guidelines and the policyvault mirror may fail at the remote server; those failures are reported honestly and are excluded from the knowledge base.

In [ ]:
corpus, ingestion = be.build_corpus(download=True)
ingestion_table = pd.DataFrame(ingestion)[["id", "organization", "status", "pages", "chunks", "chars", "reason"]]
display(ingestion_table)
indexed = ingestion_table[ingestion_table["status"] == "OK"]
failed = ingestion_table[ingestion_table["status"] == "FAIL"]
print(f"Indexed documents: {len(indexed)}")
print(f"Indexed chunks: {len(corpus):,}")
print(f"Failed or unavailable sources: {len(failed)}")
if len(failed):
    display(failed[["id", "reason"]])

## 3. Local retrieval index

Retrieval uses TF-IDF over the indexed chunks. It is local, deterministic, and does not require an external embedding service.

In [ ]:
store = be.TfidfVectorStore(corpus)
print("Store available:", store.available)
print("Chunks indexed:", len(store.chunks))
query = "fast breathing infant under 3 months pneumonia danger signs"
results = store.search(query, k=5)
for result in results:
    print(result["score"], "|", result["organization"], "|", result["title"], "| page", result["page"])
    print(result["text"][:280].replace("\n", " "))
    print()

## 4. Hand-compiled knowledge graph and investigation table

The disease graph is a transparent candidate generator, not a diagnostic model. Every graph entry carries a clinician-validation note. The investigation table contains discussion prompts, not orders.

In [ ]:
diseases = be.load_diseases()
investigations = be.load_investigations()
print(f"Disease entries: {len(diseases)}")
print(f"Investigation entries: {len(investigations)}")
display(pd.DataFrame(diseases)[["id", "name", "serious", "age_min_months", "age_max_months", "category"]].head(15))
display(pd.DataFrame(investigations)[["test", "diseases", "age_min_months", "age_max_months"]].head(10))

## 5. Deterministic safety rules

Safety rules are evaluated before retrieval and before any optional LLM call. The examples below show emergency escalation, missing-data handling, and contradiction detection.

In [ ]:
patient1 = {
    "age_value": 2,
    "age_unit": "months",
    "sex": "female",
    "weight_kg": 4.2,
    "vitals": {
        "temperature_c": 38.2,
        "respiratory_rate": 62,
        "heart_rate": 170,
        "spo2": 88,
        "bp_systolic": None,
    },
    "symptoms": ["difficulty_breathing", "fast_breathing", "poor_feeding"],
    "danger_signs": ["unable_to_feed"],
    "history": {
        "gestational_age_weeks": 35,
        "birth_weight_kg": 2.1,
        "immunisation": "Incomplete",
        "feeding_type": "Breastfeeding",
    },
    "observations": {"feeding": "poor", "urine_stool": "reduced wet nappies"},
}

profile1 = be.patient_profile(patient1)
urgency1 = be.assess_emergency(patient1)
print("Urgency:", urgency1["level"])
print("Fast-breathing threshold:", urgency1["fast_breathing_threshold"])
print("Reasons:")
for reason in urgency1["reasons"]:
    print("-", reason)
print("Missing:", profile1["missing"])
print("Contradictions:", profile1["contradictions"])
print("Safety conditions:", urgency1["safety_conditions"])

In [ ]:
neonate_fever = {
    "age_value": 1,
    "age_unit": "months",
    "sex": "male",
    "weight_kg": 3.6,
    "vitals": {"temperature_c": 38.4, "respiratory_rate": 48, "heart_rate": 160, "spo2": 97, "bp_systolic": 70},
    "symptoms": ["fever", "poor_feeding"],
    "danger_signs": [],
}
print("Neonate fever urgency:", be.assess_emergency(neonate_fever)["level"])

missing_age = {
    "sex": "female",
    "weight_kg": 8.0,
    "vitals": {"temperature_c": 36.8, "respiratory_rate": 30, "heart_rate": 110, "spo2": 98, "bp_systolic": 90},
    "symptoms": ["cough"],
    "danger_signs": [],
}
missing_result = be.assess_emergency(missing_age)
print("Missing-age urgency:", missing_result["level"])
print("Missing items:", missing_result["missing"])

contradiction_case = {
    "age_value": 10,
    "age_unit": "months",
    "sex": "male",
    "weight_kg": 8.4,
    "vitals": {"temperature_c": 37.2, "respiratory_rate": 32, "heart_rate": 110, "spo2": 98, "bp_systolic": 92},
    "symptoms": ["fever", "poor_feeding"],
    "danger_signs": ["unable_to_feed"],
    "observations": {"feeding": "feeding well"},
}
print("Contradictions:", be.find_contradictions(contradiction_case))

## 6. Optional LLM

Claude support is optional. It is enabled only when `ANTHROPIC_API_KEY` is set and `BABYCARE_DISABLE_LLM` is not true. The deterministic workflow works without it. Only a de-identified patient summary and retrieved excerpts are sent when enabled.

In [ ]:
llm = be.LLMClient()
print("LLM enabled:", llm.enabled)
print("Model:", llm.model)
print("Last error:", llm.last_error or "none")
print("De-identified patient keys:", sorted(be.deidentify_patient(patient1).keys()))

## 7. Agentic workflow

The workflow runs seven stages:

1. Patient Information Agent
2. Safety and Emergency Agent
3. Pediatric Research Agent
4. Disease Analysis Agent
5. Evidence Verification Agent
6. Medical Investigation Agent
7. Medical Report Agent

The helper below prints a compact report view.

In [ ]:
def show_report(report):
    urgency = report["urgency"]
    print("=" * 90)
    print("URGENCY:", urgency["level"])
    for reason in urgency["reasons"][:8]:
        print(" -", reason)
    print()
    print("SUMMARY:", report["summary"])
    print()
    disease_rows = [
        {
            "Condition": item["name"],
            "Status": item["evidence_status"],
            "Serious": item["serious"],
            "Why considered": ", ".join(item["why_considered"]),
            "References": len(item["references"]),
        }
        for item in report["diseases"]
    ]
    display(pd.DataFrame(disease_rows))
    if report["investigations"]:
        display(pd.DataFrame(report["investigations"])[["test", "why", "age_consideration"]])
    print("Follow-up questions:", len(report["followup_questions"]))
    print("References:", len(report["references"]))
    print("Agent stages:")
    for entry in report["agent_log"]:
        print(" -", entry["agent"], "|", entry["detail"])
    return report

## 8. Demonstration case 1: young infant with respiratory distress

This case triggers emergency rules for low SpO2, a danger sign, and age under 3 months. The graph proposes candidates; the retrieval and verification stages decide what can be shown as supported.

In [ ]:
report1 = be.run_agents(patient1, store, llm=llm)
show_report(report1)
display(Markdown(be.report_to_markdown(report1)[:5000]))

## 9. Demonstration case 2: gastroenteritis with dehydration

This case demonstrates urgent review driven by dehydration symptoms and age-specific vital signs. The report separates graph candidates from retrieved evidence.

In [ ]:
patient2 = {
    "age_value": 18,
    "age_unit": "months",
    "sex": "male",
    "weight_kg": 10.2,
    "vitals": {
        "temperature_c": 38.5,
        "respiratory_rate": 34,
        "heart_rate": 138,
        "spo2": 97,
        "bp_systolic": 88,
    },
    "symptoms": ["diarrhea", "vomiting", "dehydration", "poor_feeding"],
    "danger_signs": [],
    "history": {"immunisation": "Complete", "feeding_type": "Mixed"},
    "observations": {"feeding": "poor", "urine_stool": "fewer wet nappies and watery stools"},
    "duration_days": 3,
}
report2 = be.run_agents(patient2, store, llm=llm)
show_report(report2)

## 10. Demonstration case 3: missing information is not normal

This case deliberately omits systolic blood pressure. The safety layer raises the urgency and lists the missing measurement rather than treating it as normal.

In [ ]:
patient3 = {
    "age_value": 4,
    "age_unit": "years",
    "sex": "female",
    "weight_kg": 16.0,
    "vitals": {
        "temperature_c": 39.2,
        "respiratory_rate": 26,
        "heart_rate": 118,
        "spo2": 98,
        "bp_systolic": None,
    },
    "symptoms": ["fever", "cough", "poor_feeding"],
    "danger_signs": [],
    "history": {"immunisation": "Complete", "feeding_type": "Mixed"},
    "observations": {"feeding": "reduced", "urine_stool": "normal urine output"},
    "duration_days": 2,
}
report3 = be.run_agents(patient3, store, llm=llm)
show_report(report3)
print("Missing items:", report3["profile"]["missing"])

## 11. Follow-up update

Follow-up answers are merged into the structured patient record and the full workflow reruns. The original report is not silently overwritten; this notebook keeps both report objects.

In [ ]:
questions = report1["followup_questions"]
print("Follow-up questions:")
for question in questions:
    print("-", question["id"], "|", question["question"])

answers = {
    "vital_bp_systolic": "72",
    "history_feeding_type": "Breastfeeding",
}
updated_patient1 = be.apply_followup_answers(patient1, answers)
report1_updated = be.run_agents(updated_patient1, store, llm=llm)
print("Original urgency:", report1["urgency"]["level"])
print("Updated urgency:", report1_updated["urgency"]["level"])
print("Original missing:", report1["profile"]["missing"])
print("Updated missing:", report1_updated["profile"]["missing"])
show_report(report1_updated)

## 12. Safety and reference-audit tests

These assertions exercise mandatory safety behaviors and verify that every displayed reference came from an indexed source manifest entry.

In [ ]:
def assert_urgency(patient, expected):
    result = be.assess_emergency(patient)
    assert result["level"] == expected, (result["level"], result["reasons"])
    return result

assert_urgency(patient1, "Emergency")
assert_urgency(neonate_fever, "Emergency")
assert_urgency(missing_age, "Urgent")

assert "Exact age in months" in missing_result["missing"]
assert "Systolic BP" in report3["profile"]["missing"]
assert be.find_contradictions(contradiction_case)
assert all("No indexed passage" not in item for item in report1["diseases"][0]["information_needed"])

source_urls = {source["url"] for source in sources}
for report in [report1, report2, report3, report1_updated]:
    for reference in report["references"]:
        assert reference["url"] in source_urls, reference["url"]
        assert reference["title"] and reference["page"] is not None

for report in [report1, report2, report3, report1_updated]:
    markdown = be.report_to_markdown(report)
    assert "http://" not in markdown.replace("https://", "")
    assert "### " in markdown
    assert report["disclaimer"]

print("All safety and reference-audit assertions passed.")

## 13. Save demonstration reports

Reports are written as Markdown with identifiers withheld. These files are examples only and must not be treated as clinical records.

In [ ]:
out_dir = Path("data/demo_reports")
out_dir.mkdir(parents=True, exist_ok=True)
for name, report in [
    ("case1_young_infant_respiratory", report1),
    ("case2_gastroenteritis_dehydration", report2),
    ("case3_missing_information", report3),
    ("case1_updated_followup", report1_updated),
]:
    path = out_dir / f"{name}.md"
    path.write_text(be.report_to_markdown(report), encoding="utf-8")
    print(path, path.stat().st_size, "bytes")

## 14. Limitations and governance

- This is a prototype clinical decision-support tool, not a medical device.
- The disease graph and investigation table are hand-compiled and require clinician validation.
- Evidence quality depends on the indexed documents. Failed downloads are reported and excluded.
- Retrieval can miss relevant passages. A missing passage does not mean a condition is absent.
- The tool does not provide drug names, doses, or treatment regimens.
- The optional LLM must not be used with identifiable patient data.
- A clinician remains responsible for assessment, testing, treatment, and escalation.
- Any deployment requires local clinical governance, validation, security review, and data-protection review.

The Rwanda 2023 paediatric guideline and Basic Paediatric Protocol uploads failed at the RBC server because the filenames contain a special character. The policyvault mirror also failed to connect. Those sources are therefore not represented as indexed evidence.